# FLOPs-budgeted HPO

ASHA searches learning rate and weight decay for the selected NATS-Bench architectures. Reusable experiment code lives in `kd_vs_hpo.hpo`.

In [ ]:
from pathlib import Path

import torch
from IPython.display import HTML, display

from kd_vs_hpo.common import TrainConfig
from kd_vs_hpo.hpo import ASHAConfig, HPOExperimentConfig, SearchSpace, run_hpo_experiment

%load_ext autoreload
%autoreload 2

In [ ]:
experiment = HPOExperimentConfig(
    train=TrainConfig(
        batch_size=256,
        num_workers=2,
        validation_fraction=0.1,
        momentum=0.9,
        grad_clip_norm=5.0,
        seed=42,
        deterministic=False,
        amp=True,
        train_step_multiplier=3.0,
        data_root=Path("data"),
    ),
    search_space=SearchSpace(
        lr=(1e-3, 3e-1),
        weight_decay=(1e-6, 1e-3),
    ),
    asha=ASHAConfig(
        budget_flops_per_arch=10**15,
        target_min_epochs=3,
        reduction_factor=3,
        max_initial_configs=12,
        max_epochs=81,
    ),
    architectures_path=Path("experiments/nats_architectures_10.json"),
    costs_path=Path("experiments/sampled_architecture_costs.csv"),
    output_dir=Path("hpo_output"),
    arch_rows=(0,),  # None runs all architectures.
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
result = run_hpo_experiment(experiment, device)
display(result.summary)
result.plot_paths

In [ ]:
for plot_path in result.plot_paths:
    display(HTML(f'<a href="{plot_path}" target="_blank">Open {plot_path.name}</a>'))